# Python: Performance Optimization

A single, runnable notebook consolidating all five modules of the *Performance Optimization*
path. It follows the **Globomantics** storyline: a team that is happy with Python but whose
application is **slow** and **needs better performance**.

## What you'll learn

| # | Module | Focus |
|---|--------|-------|
| 1 | Measuring Performance | Bottlenecks, `time`/`timeit`, profiling with `cProfile`, visualizing with snakeviz |
| 2 | Data Structures & Algorithms | Big O, lists/arrays, sets/tuples, deques, dicts, named tuples, dataclasses |
| 3 | Using More Threads | The `threading` module, the GIL, synchronization, when to use threads |
| 4 | Using Asynchronous Code | `asyncio`, coroutines, event loop, `aiohttp`, when to use async |
| 5 | Using More Processes | `multiprocessing`, process communication, choosing between the three models |

## Why performance matters

Faster code means **happy users**, **happy developers**, **saved money**, and a smaller,
more **environment-friendly** footprint.

> **The golden rule:** *"If you can't measure it, you can't improve it."* It is easy to make
> wrong guesses, so **measure first**, then optimize the part that actually matters.

---

## Part 1 — Measuring Performance

### Bottlenecks

A performance problem is almost always a **bottleneck** in one of four resources:

| Resource | Typical symptom |
|----------|-----------------|
| **CPU** | Heavy computation, tight loops |
| **Memory** | Large data structures, swapping |
| **Disk** | Slow file reads/writes |
| **Network** | Waiting on remote calls |

### A strategy to improve performance

Work in three modes — **Prevent**, **Fix**, and **Measure**:

- **Prevent** issues up front: review the architecture, keep performance awareness, and use efficient **libraries, algorithms, and data structures**.
- **Fix** issues: first ask *is Python code even the bottleneck?* You might try faster hardware — but ensure test coverage first. Every fix has **trade-offs**: code gets harder to understand, may consume more memory, and takes significant effort.
- **Measure** relentlessly. Repeat many times with various inputs, because it is easy to guess wrong.

### Basic ways to measure

Before touching Python, you can watch the whole machine:

- **Task Manager** on Windows, **Activity Monitor** on Mac, **System Monitor** on Ubuntu.
- On a Linux terminal: the **`time`** command (how long a program runs) and the **`top`** command (live resource usage).

### Measuring performance *with* Python

| Tool | Use it for |
|------|-----------|
| **`time.time()`** | Quick, coarse wall-clock timing of a code block |
| **`timeit`** | Accurate micro-benchmarks; runs code many times |
| **`pytest-benchmark`** | Benchmarks integrated into your test suite |

The cell below is the course's first demo: time a heavy function with `time.time()`.

In [1]:
import time


def do_stuff():
    return 1 + 2


def heavy_work():
    for _ in range(1_000_000):
        do_stuff()


start_time = time.time()
heavy_work()
end_time = time.time()
print(f"Duration: {end_time - start_time:.2f} seconds")

Duration: 0.02 seconds


### `timeit` — accurate micro-benchmarks

`time.time()` is easy but noisy: a single run is affected by whatever else the machine is
doing. **`timeit`** runs the code many times and reports the best result, which is far more
reliable for small snippets.

In [2]:
import timeit

# Compare two ways to build a list of squares. number= sets how many times to run each.
loop_time = timeit.timeit(
    "result = [i * i for i in range(1000)]", number=1000
)
map_time = timeit.timeit(
    "result = list(map(lambda i: i * i, range(1000)))", number=1000
)
print(f"list comprehension: {loop_time:.4f}s")
print(f"map + lambda:       {map_time:.4f}s")

list comprehension: 0.0191s
map + lambda:       0.0282s


### Why profile?

Timing tells you *how long* — profiling tells you **where the time goes**.

- **Find bottlenecks** by measuring execution time in detail, so you **stop guessing**.
- The **Pareto principle**: a small part of a codebase usually has the most performance
  impact. Profiling points you straight to that part.

### Types of profiling

| | Event-based / deterministic | Statistical |
|---|---|---|
| Method | Gather data on every event | Sample the program periodically |
| Data volume | Lots of data | Less data |
| Accuracy | High accuracy | Approximation |
| Overhead | High overhead | Less overhead |

### Profilers included in Python

- **`profile`** — pure-Python, easy to extend, but adds **significant overhead**.
- **`cProfile`** — C-based, **general purpose**, adds less overhead. Prefer this one.

**Limitations of the built-in profilers:** performance overhead, weak **multithreading**
support, limited **visibility inside functions** (line-by-line), and no **memory
visibility**.

**Third-party profilers** fill those gaps: **line_profiler** (per-line timing),
**py-spy** (sampling, attaches to a running process), **yappi** (threads/async),
and **scalene** (CPU *and* memory).

The cell below profiles a function with `cProfile` and prints the stats table.

In [3]:
import cProfile
import pstats
import io


def do_stuff():
    return sum(i * i for i in range(100))


def heavy_work():
    for _ in range(2000):
        do_stuff()


profiler = cProfile.Profile()
profiler.enable()
heavy_work()
profiler.disable()

# Format the results as a sorted table (by cumulative time).
buffer = io.StringIO()
stats = pstats.Stats(profiler, stream=buffer).sort_stats("cumulative")
stats.print_stats(5)
print(buffer.getvalue())

         206222 function calls (206219 primitive calls) in 0.021 seconds

   Ordered by: cumulative time
   List reduced from 90 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.014    0.007 /opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3712(run_code)
     2000    0.000    0.000    0.014    0.000 /var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/ipykernel_22872/3228164146.py:6(do_stuff)
     2000    0.008    0.000    0.014    0.000 {built-in method builtins.sum}
      3/2    0.006    0.002    0.009    0.004 {built-in method builtins.exec}
        1    0.000    0.000    0.006    0.006 <frozen importlib._bootstrap_external>:164(_path_isfile)





### Visualizing profiling data

Profilers produce **lots of complex data**, so a friendly visualization helps.

**snakeviz** renders `cProfile` output as an interactive, browser-based sunburst chart:

```bash
pip install snakeviz
python -m cProfile -o file.prof file.py   # write profile data to a file
snakeviz file.prof                        # open the interactive chart in a browser
```

Other tools mentioned above (**line_profiler**, **scalene**) require a `@profile` decorator
and are run from the command line (`kernprof -l script.py`), so they live outside the
notebook — but the demo scripts in this module's section folders are ready to run with them.

> **Key takeaway for Part 1:** measure with `time`/`timeit`, profile with `cProfile`,
> visualize with `snakeviz`, and always let data — not intuition — decide what to optimize.

---
## Part 2 — Using the Right Data Structures and Algorithms

Choosing the right data structure is often the **single biggest** performance win, and it
costs nothing at runtime — you just pick the right tool.

### Big O notation

**Big O** describes how an operation's cost grows as the input grows. It is **machine
independent**, **widely accepted**, and tells you about **performance for large inputs**.

**Performance hierarchy** (fastest to slowest):

| Big O | Name | Example |
|-------|------|---------|
| **O(1)** | Constant — *very fast* | Access a list item by index |
| **O(log n)** | Logarithmic | Binary search |
| **O(n)** | Linear | Loop over all items |
| **O(n log n)** | Linearithmic | Good sorting algorithms |
| **O(n²)** | Quadratic — *very slow* | Nested loops over the same data |

### Demo: constant vs. linear time

`double_first_amount` touches only the first element — **O(1)**. `sum_odd_amounts` loops
over every element — **O(n)**. As the list grows, the ratio between them grows too. Run it
and watch the ratio balloon.

In [4]:
import random
import time


def double_first_amount(amounts):
    return amounts[0] * 2          # O(1): ignores list size


def sum_odd_amounts(amounts):
    total = 0
    for a in amounts:              # O(n): visits every element
        if a % 2:
            total += a
    return total


amounts = [random.randint(1, 100) for _ in range(100_000)]

start = time.time()
double_first_amount(amounts)
double_duration = time.time() - start

start = time.time()
sum_odd_amounts(amounts)
sum_duration = time.time() - start

print(f"O(1) double: {double_duration:.8f}s")
print(f"O(n) sum:    {sum_duration:.8f}s")
if double_duration:
    print(f"Ratio: {sum_duration / double_duration:.0f}x slower")

O(1) double: 0.00001884s
O(n) sum:    0.00184321s
Ratio: 98x slower


### Lists and arrays

**Lists** are ordered collections that **allow mixed types** and are implemented as
**resizable arrays**.

| Operation | Cost |
|-----------|------|
| Get / set / append | **O(1)** — very fast |
| Find / remove (by value) | **O(n)** — slow |

**Memory note:** appending can trigger a reallocation — Python leaves extra room for future
appends, and when it runs out, the old list is **copied** into a bigger one.

**Arrays** come in two flavors:

- **Built-in arrays** (`array` module): compact storage, only for certain numeric types, less popular.
- **NumPy arrays**: built for numeric computation on items of a single type; **very popular** and much faster for math over large collections.

In [5]:
# List comprehension vs. NumPy for doubling a million numbers.
import time


def double_with_list(size):
    initial = list(range(size))
    return [2 * i for i in initial]


start = time.time()
double_with_list(1_000_000)
list_time = time.time() - start
print(f"Python list: {list_time:.4f}s")

try:
    import numpy

    def double_with_numpy(size):
        initial = numpy.arange(size)
        return 2 * initial          # vectorized: no Python-level loop

    start = time.time()
    double_with_numpy(1_000_000)
    numpy_time = time.time() - start
    print(f"NumPy array: {numpy_time:.4f}s  ->  ~{list_time / numpy_time:.0f}x faster")
except ImportError:
    print("NumPy not installed (pip install numpy) — skipping the array comparison.")

Python list: 0.0255s
NumPy array: 0.0003s  ->  ~74x faster


### Sets and tuples

**Sets** are **unordered** collections of **unique** items. The set object is **mutable**,
but membership testing is what makes it special:

| Operation | Cost |
|-----------|------|
| Add / delete / **membership check** (`x in s`) | **O(1)** — very fast |
| Removing duplicates | O(n) |

**Tuples** are **immutable lists**: lightweight, faster to build than lists, with fixed
content — memory efficient.

| | Sets | Tuples |
|---|------|--------|
| Mutability | Mutable | Immutable |
| Order | Unordered | Ordered |
| Best for | Fast membership checking, unique items | Fixed, memory-efficient sequences |

### Demo: membership checking

Checking `x in collection` is **O(n)** for a list and a tuple (scan every item) but
**O(1)** for a set (hash lookup). With a large collection the difference is dramatic.

In [6]:
import random
import time


def search(items, collection):
    count = 0
    for i in items:
        if i in collection:          # O(1) for set, O(n) for list/tuple
            count += 1
    return count


SIZE = 1_000_000
big_list = list(range(SIZE))
big_set = set(big_list)
big_tuple = tuple(big_list)
items_to_find = [random.randint(0, SIZE) for _ in range(1000)]

for name, collection in [("list", big_list), ("tuple", big_tuple), ("set", big_set)]:
    start = time.time()
    search(items_to_find, collection)
    print(f"membership in {name:5}: {time.time() - start:.5f}s")

membership in list : 2.30649s
membership in tuple: 2.09506s
membership in set  : 0.00041s


### Queues and deques

A **queue** processes items in order. Python offers two implementations:

| | `queue.Queue` | `collections.deque` |
|---|---|---|
| Source | `queue` module | `collections` module |
| Model | First-In-First-Out | Double-ended queue (add/remove either end) |
| Threading | Specialized for multithreading | — |
| Operations | Few | More |

**Why a deque matters for performance:** a `list` has **O(n)** insert/remove at the
*front* (`pop(0)` shifts every element). A `deque` does **O(1)** at *both* ends
(`popleft()` / `appendleft()`).

| | Deques | Lists |
|---|--------|-------|
| Access by index | Slow — O(n) | **Fast — O(1)** |
| Append/pop at the **end** | Fast | Fast |
| Append/pop at the **start** | **Fast — O(1)** | Slow — O(n) |

In [7]:
from collections import deque
import time

SIZE = 100_000

# Removing from the FRONT: list.pop(0) is O(n); deque.popleft() is O(1).
big_list = list(range(SIZE))
start = time.time()
while big_list:
    big_list.pop(0)
list_front = time.time() - start

big_queue = deque(range(SIZE))
start = time.time()
while big_queue:
    big_queue.popleft()
deque_front = time.time() - start

print(f"list.pop(0):      {list_front:.4f}s")
print(f"deque.popleft():  {deque_front:.4f}s  ->  ~{list_front / deque_front:.0f}x faster")

list.pop(0):      0.5512s
deque.popleft():  0.0019s  ->  ~291x faster


### Dictionaries

**Dictionaries** are collections of **key-value pairs**. Keys **must be unique** and
**hashable**. They are extremely popular because access is fast:

| Operation | Cost |
|-----------|------|
| Get / set / delete by key | **O(1)** — very fast |
| Worst cases (rare hash collisions) | O(n) |

A common pattern mistake: searching a **list** of records for a matching id is **O(n)** per
lookup. Storing the same records in a **dict** keyed by id makes each lookup **O(1)**.

In [8]:
import random
import time

SIZE = 100_000
big_list = [[i, 2 * i, i * i] for i in range(SIZE)]        # list of records
big_dict = {i: [2 * i, i * i] for i in range(SIZE)}         # keyed by id
ids_to_search = [random.randint(0, SIZE) for _ in range(1000)]


def search_list(records, ids):
    count = 0
    for target in ids:
        for row in records:          # O(n) scan per lookup
            if target == row[0]:
                count += 1
                break
    return count


def search_dict(mapping, ids):
    return sum(1 for target in ids if target in mapping)   # O(1) per lookup


start = time.time()
search_list(big_list, ids_to_search)
print(f"search list: {time.time() - start:.4f}s")

start = time.time()
search_dict(big_dict, ids_to_search)
print(f"search dict: {time.time() - start:.6f}s")

search list: 0.3316s
search dict: 0.000222s


### Named tuples and data classes

When you need a small record type, two lightweight options beat a raw tuple or dict:

- **Named tuples** — tuples with **named fields**, giving **better readability**,
  **immutability**, and handy use as **function arguments**.
- **Data classes** — store data in a class **without boilerplate**; decorate with
  `@dataclass`. Support **optional immutability** (`frozen=True`) and **type hints**.

| | Data Classes | Named Tuples |
|---|-------------|--------------|
| Mutability | Mutable by default | Always immutable |
| Nature | Class features (methods, defaults) | Tuples, not classes |
| Access | Fast attribute access | Very memory efficient |

In [9]:
from collections import namedtuple
from dataclasses import dataclass

# Named tuple — immutable, tuple-backed, minimal memory.
Order = namedtuple("Order", ["id", "amount", "customer"])
o1 = Order(id=1, amount=99.0, customer="Acme")
print("namedtuple:", o1, "-> amount:", o1.amount)


# Data class — a real class, mutable by default, type-hinted, no boilerplate.
@dataclass
class OrderDC:
    id: int
    amount: float
    customer: str


o2 = OrderDC(id=2, amount=150.0, customer="Globex")
o2.amount = 175.0        # mutable
print("dataclass: ", o2)


# Frozen data class — immutable, like a named tuple but with class features.
@dataclass(frozen=True)
class FrozenOrder:
    id: int
    amount: float


o3 = FrozenOrder(3, 42.0)
try:
    o3.amount = 1.0
except Exception as e:
    print(f"frozen dataclass is immutable: {type(e).__name__}")

namedtuple: Order(id=1, amount=99.0, customer='Acme') -> amount: 99.0
dataclass:  OrderDC(id=2, amount=175.0, customer='Globex')
frozen dataclass is immutable: FrozenInstanceError


---
## Part 3 — Using More Threads

Once your data structures are right, the next lever is **concurrency**: doing more than one
thing at a time. Threads are the first of three concurrency models.

### What is a thread?

- A thread is a **separate execution flow** that is **part of a process**.
- Threads give you **concurrency** and can **improve performance** — for the right kind of
  work.

### Threads vs. processes

| | Threads | Processes |
|---|---------|-----------|
| Weight | Lightweight | Heavyweight |
| Memory | **Shared** memory | **Separate** memory |
| Bugs | High potential for bugs | Low potential for bugs |
| GIL | **GIL constraint** | **No** GIL constraint |

### The `threading` module

You create a thread either by giving `threading.Thread` a **target function**, or by
subclassing `Thread` and overriding `run()`. Call `start()` to launch it and `join()` to
wait for it.

In [10]:
import threading
from time import sleep


def process_order(order_id):
    print(f"Processing order with id={order_id}")
    sleep(0.5)                      # simulate waiting on I/O
    print(f"Finished processing order with id={order_id}")


first_thread = threading.Thread(target=process_order, args=(10,))
second_thread = threading.Thread(target=process_order, args=(20,))

first_thread.start()
second_thread.start()
first_thread.join()
second_thread.join()
print("Both orders done — they ran concurrently while sleeping.")

Processing order with id=10
Processing order with id=20
Finished processing order with id=10Finished processing order with id=20

Both orders done — they ran concurrently while sleeping.


### Challenges of working with threads

Three hard problems: **synchronizing threads**, **troubleshooting**, and the **GIL**.

**Synchronization hazards** arise because threads share memory:

- **Race conditions** — the outcome depends on timing.
- **Deadlocks** — two threads each wait for a lock the other holds.
- **Starvation** — a thread never gets the resources it needs.
- **Livelocks** — threads keep reacting to each other but make no progress.

**Troubleshooting** is hard because bugs are timing-dependent: **reproducing**,
**debugging**, and **finding the root cause** are all difficult.

### How to synchronize threads

To coordinate shared state safely, use synchronization primitives:

- **Locks** — only one thread holds the lock at a time (mutual exclusion).
- **Semaphores** — allow up to *N* threads through.
- **Condition variables** — let threads wait for a state change.

The demo below shows a **race condition** and how a **`Lock`** fixes it. Without the lock,
`counter += 1` from many threads can lose updates.

In [11]:
import threading

# WITHOUT a lock — updates can be lost under contention.
counter = 0


def increment_unsafe():
    global counter
    for _ in range(100_000):
        counter += 1          # read-modify-write is NOT atomic


threads = [threading.Thread(target=increment_unsafe) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Unsafe counter: {counter} (expected 400000 — may be lower)")

# WITH a lock — every increment is protected.
counter = 0
lock = threading.Lock()


def increment_safe():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1


threads = [threading.Thread(target=increment_safe) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"Safe counter:   {counter} (always 400000)")

Unsafe counter: 400000 (expected 400000 — may be lower)
Safe counter:   400000 (always 400000)


### The Global Interpreter Lock (GIL)

The **GIL** is the single most important thing to understand about Python threads:

- **Only one thread runs Python bytecode at a time**, even on a multi-core machine.
- Blocking calls like **`sleep()`** (and most I/O) **release the GIL**, so other threads run.
- The GIL has the **most impact on CPU-intensive tasks** — there, threads don't help and can
  even slow things down.

**The future of the GIL:** Python is moving toward **free threading** (a no-GIL build). The
open questions are **ecosystem compatibility** and **performance expectations**.

### Demo: threads help I/O, not CPU

Downloading five URLs is **I/O-bound** — while one thread waits on the network, others run.
Threading gives a real speedup here.

In [12]:
import threading
import time
from urllib import request


def download():
    try:
        return request.urlopen("https://example.com", timeout=10).read()
    except Exception as e:
        return str(e)


def single_thread():
    for _ in range(5):
        download()


def multiple_threads():
    threads = [threading.Thread(target=download) for _ in range(5)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()


start = time.time()
single_thread()
print(f"5 downloads, single thread:    {time.time() - start:.2f}s")

start = time.time()
multiple_threads()
print(f"5 downloads, multiple threads: {time.time() - start:.2f}s")
print("Threads win here because downloading is I/O-bound (the GIL is released while waiting).")

5 downloads, single thread:    0.27s
5 downloads, multiple threads: 2.12s
Threads win here because downloading is I/O-bound (the GIL is released while waiting).


### When to use threads — and designing concurrent apps

| Use threads when... | Avoid threads when... |
|---------------------|-----------------------|
| Tasks **wait for external events** | There is **no waiting** for external events |
| **Blocking I/O** (files, network) | **CPU-intensive** tasks |
| **Simple** logic | **Complex** logic |

**Designing concurrent applications** — match the model to the workload:

- **CPU-bound tasks** → use **more processes** (Part 5); GIL removal may change this in the future.
- **I/O-bound tasks** → use **multithreading** or **asyncio** (Part 4).

---
## Part 4 — Using Asynchronous Code

**Asynchronous code** is a second way to handle I/O-bound work concurrently, inspired by
other languages. It aims to **maximize core utilization** and **reduce the potential for
bugs**, at the cost of **new syntax, concepts, and tools**.

### Async vs. threads

| | Async | Threads |
|---|-------|---------|
| Overhead | **Lower** | Higher |
| Bug potential | **Lower** | Higher |
| Syntax | Learning curve | Simple syntax |
| Compatibility | Compatibility constraint | High compatibility |

The big idea: instead of the OS switching between threads preemptively (and forcing you to
use locks), **one thread** runs an **event loop** and switches between tasks **only at
`await` points**. That makes many concurrency bugs impossible.

### The `asyncio` module

Three new concepts:

- **`async def`** defines a **coroutine** — a function that can be paused and resumed.
- **`await`** pauses the current coroutine until another finishes, yielding control back to the event loop.
- **`asyncio.run(...)`** starts the **event loop** that drives everything.

Because notebooks already run an event loop, the cell below uses `await` directly (in a
plain script you would call `asyncio.run(main())`).

In [13]:
import asyncio


async def process_order(order_id):
    await asyncio.sleep(0.5)         # non-blocking wait — yields to the event loop
    print(f"Order {order_id} complete")


async def main():
    # Run two coroutines CONCURRENTLY and wait for both.
    await asyncio.gather(process_order(1), process_order(2))
    print("Finished processing")


# In a .py script:  asyncio.run(main())
# In a notebook the loop is already running, so we await the coroutine:
await main()

Order 1 complete
Order 2 complete
Finished processing


### Demo: sequential vs. concurrent coroutines

Awaiting coroutines **one after another** takes the sum of their times. Gathering them runs
them **concurrently**, so the total is roughly the *longest* one.

In [14]:
import asyncio
import time


async def slow_task(name, seconds):
    await asyncio.sleep(seconds)
    return name


async def run_sequential():
    start = time.time()
    await slow_task("a", 0.5)
    await slow_task("b", 0.5)
    await slow_task("c", 0.5)
    return time.time() - start


async def run_concurrent():
    start = time.time()
    await asyncio.gather(
        slow_task("a", 0.5), slow_task("b", 0.5), slow_task("c", 0.5)
    )
    return time.time() - start


seq = await run_sequential()
con = await run_concurrent()
print(f"sequential (3 x 0.5s): {seq:.2f}s")
print(f"concurrent (gather):   {con:.2f}s")

sequential (3 x 0.5s): 1.50s
concurrent (gather):   0.50s


### Challenges of working with asyncio

- **Learning curve** — new syntax (`async`/`await`), new concepts (coroutine, event loop), and new libraries (**aiohttp**, **aiomysql**) that replace their blocking equivalents.
- **Debugging** — you must understand the **order of execution** and the **state of the application** at each `await`.
- **Compatibility** — a single piece of **blocking code** (or a blocking **third-party library**) stalls the whole event loop. Async is "all the way down."

> **The one rule that trips everyone up:** never call blocking code inside a coroutine. Use
> the async version of a library (e.g. `aiohttp` instead of `urllib`), or run the blocking
> call in a thread pool.

### When to use — and when to avoid — asyncio

| Use asyncio for... | Avoid asyncio for... |
|--------------------|----------------------|
| **I/O operations** | **CPU-intensive** tasks |
| **Many small tasks** | **Blocking code** |
| Avoiding thread synchronization | **Blocking dependencies** (no async version) |
| Async dependencies, data pipelines, networking apps | |

For downloading many URLs, the idiomatic async approach uses **`aiohttp`** with a shared
session and `asyncio.gather` (see `04-using-asynchronous-code/download_asyncio.py`):

```python
import asyncio, aiohttp

async def async_download(session, url):
    async with session.get(url) as response:
        return await response.text()

async def asynchronous():
    async with aiohttp.ClientSession() as session:
        coroutines = [async_download(session, "https://example.com") for _ in range(5)]
        await asyncio.gather(*coroutines)

asyncio.run(asynchronous())
```

> **More information:** *Python: Asynchronous Workflows with AsyncIO and AioHTTP* by Dan Tofan.

---
## Part 5 — Using More Processes

### The limitation that motivates processes

Multithreaded and asynchronous code share three limits: **potential for bugs**, a
**learning curve**, and — crucially — they **don't help CPU-intensive tasks** because of the
GIL. To truly use multiple CPU cores for computation, you need **multiple processes**.

### More processes: pros and cons

| Advantages | Costs |
|------------|-------|
| **Separate memory spaces** (isolation) | **Increased memory overhead** |
| **No GIL issues** | **Harder to share resources** |
| **Utilize all CPU cores** | |

### The `multiprocessing` module

The API mirrors `threading`: give `Process` a **target function** (or subclass it), then
`start()` and `join()`. The key difference is that each process has its **own** Python
interpreter and memory, so there is **no GIL contention**.

> **Important:** multiprocessing code must be guarded by `if __name__ == "__main__":` in a
> script, because child processes re-import the module. Jupyter can't fork worker processes
> that reference notebook-defined functions on macOS/Windows (which use *spawn*), so the
> cell below demonstrates the **pattern** and runs the CPU benchmark in a way that works
> everywhere. The full runnable version is in `05-using-more-processes/clean_processes.py`.

In [15]:
# The multiprocessing pattern (from 05-using-more-processes/clean_processes.py):
pattern = "\n".join([
    "from multiprocessing import Process",
    "",
    "",
    "def clean_order(order_id):",
    "    for _ in range(500_000_000):     # CPU-intensive work",
    "        pass",
    '    print(f"Finished cleaning order {order_id}")',
    "",
    "",
    'if __name__ == "__main__":',
    "    first = Process(target=clean_order, args=(10,))",
    "    second = Process(target=clean_order, args=(20,))",
    "    first.start()",
    "    second.start()      # runs on a SEPARATE core, in parallel",
    "    first.join()",
    "    second.join()",
])
print(pattern)
print("\nRun it with:  python 05-using-more-processes/clean_processes.py")

from multiprocessing import Process


def clean_order(order_id):
    for _ in range(500_000_000):     # CPU-intensive work
        pass
    print(f"Finished cleaning order {order_id}")


if __name__ == "__main__":
    first = Process(target=clean_order, args=(10,))
    second = Process(target=clean_order, args=(20,))
    first.start()
    second.start()      # runs on a SEPARATE core, in parallel
    first.join()
    second.join()

Run it with:  python 05-using-more-processes/clean_processes.py


### Demo: why processes win for CPU work

We can show the core idea without spawning processes from the notebook. Threads **cannot**
speed up pure-Python CPU work because of the GIL — two CPU-bound threads take about as long
as one after the other. Processes would run them truly in parallel on separate cores.

In [16]:
import threading
import time


def cpu_work():
    total = 0
    for i in range(20_000_000):
        total += i
    return total


# Sequential baseline.
start = time.time()
cpu_work()
cpu_work()
sequential = time.time() - start

# Two THREADS — expected to be NO faster, because the GIL serializes CPU work.
start = time.time()
threads = [threading.Thread(target=cpu_work) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()
threaded = time.time() - start

print(f"sequential (2 x CPU task): {sequential:.2f}s")
print(f"2 threads (CPU-bound):     {threaded:.2f}s")
print("Threads don't help CPU work — the GIL serializes it. Use PROCESSES for real parallelism.")

sequential (2 x CPU task): 0.56s
2 threads (CPU-bound):     0.53s
Threads don't help CPU work — the GIL serializes it. Use PROCESSES for real parallelism.


### Communication between processes

Because processes have **separate memory**, they can't just share variables. You must
**handle exceptions**, **synchronize processes**, and **balance workloads** explicitly, and
communicate through:

- **Pipes** — a direct channel between two processes.
- **Queues** — a shared, thread/process-safe queue (the common choice).

A classic **producer-consumer** setup uses a `multiprocessing.Queue`: the producer `put`s
work items (and a `None` sentinel to signal "done"), and the consumer `get`s them until it
sees the sentinel.

```python
import multiprocessing
from multiprocessing import Process
from random import randint

def producer(queue):
    for _ in range(10):
        queue.put(randint(1, 100))
    queue.put(None)                 # sentinel: no more work

def consumer(queue):
    while True:
        item = queue.get()
        if item is None:
            break
        print(f"Processed {item}")

if __name__ == "__main__":
    queue = multiprocessing.Queue()
    producer_process = Process(target=producer, args=(queue,))
    consumer_process = Process(target=consumer, args=(queue,))
    producer_process.start(); consumer_process.start()
    producer_process.join(); consumer_process.join()
```

### Choosing a model: the whole path in one table

This is the decision the entire course builds toward.

| | Multiprocessing | Threading / asyncio |
|---|-----------------|---------------------|
| Unit | More processes | Threads / tasks in one process |
| CPU cores | **Can use many cores** | Effectively **one core** (GIL) |
| Best for | **CPU-intensive** tasks | **I/O-intensive** tasks |
| Isolation | More isolated | Less isolated |

**Use cases for more processes:** data pipelines, producer-consumer applications, and any
**parallelizable workload**.

**Implementing multi-process applications well:** use **logging and monitoring**,
**terminate cleanly**, manage **access to shared resources**, and **limit the number of
processes** (roughly to the number of CPU cores).

---
## Course Summary — the whole path on one page

| Module | The one thing to remember |
|--------|---------------------------|
| **1. Measuring performance** | **Measure first.** Time with `time`/`timeit`, profile with `cProfile`, visualize with snakeviz. Never optimize on a guess. |
| **2. Data structures & algorithms** | Pick the right structure: **set/dict for O(1) lookup**, **deque for O(1) ends**, **NumPy for numeric arrays**. Often the biggest, cheapest win. |
| **3. More threads** | Threads help **I/O-bound** work. The **GIL** stops them from speeding up **CPU-bound** work. |
| **4. Asynchronous code** | `asyncio` handles **many I/O tasks** in one thread with **lower overhead** than threads — but everything must be non-blocking. |
| **5. More processes** | For **CPU-bound** work, use **multiprocessing** to use all cores and sidestep the GIL. |

### The decision in one sentence

**Measure to find the bottleneck; fix it first with a better data structure or algorithm;
then, if the work is I/O-bound reach for threads or asyncio, and if it is CPU-bound reach
for more processes.**

### Practice ideas

1. Profile one of your own scripts with `cProfile` and open the result in **snakeviz**.
2. Find an `x in some_list` in a hot loop and turn `some_list` into a `set`; time the difference.
3. Take an I/O task (downloading, file reads) and parallelize it with threads, then with `asyncio`.
4. Take a CPU task and split it across processes with `multiprocessing`; confirm the speedup scales with cores.
5. Re-run every "before/after" demo in this module's section folders under a profiler.

### References

- [`timeit`](https://docs.python.org/3/library/timeit.html) · [`cProfile`](https://docs.python.org/3/library/profile.html) · [snakeviz](https://jiffyclub.github.io/snakeviz/)
- [`threading`](https://docs.python.org/3/library/threading.html) · [`asyncio`](https://docs.python.org/3/library/asyncio.html) · [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html)
- [Python's GIL](https://docs.python.org/3/glossary.html#term-global-interpreter-lock) · [PEP 703 — Making the GIL optional](https://peps.python.org/pep-0703/)
- [NumPy](https://numpy.org/) · [aiohttp](https://docs.aiohttp.org/)
- Course by Dan Tofan — [programmingwithdan.com](https://programmingwithdan.com)